# QuantumMedx — QFF 2026 QML Track

This notebook documents the Q2 classification experiments and verifies the
official Task A and Task B benchmarks. Run the cells from the project root
with the `.venv-qff` environment.

**Important:** Task A results are calculated by the official benchmark code.
This notebook does not invent benchmark values or use hidden test labels.


## 1. Check the project files

In [1]:
from pathlib import Path

root = Path.cwd()
required = [
    "train_q2.py",
    "predictions_q2.csv",
    "processor_d_config.json",
    "official_kit/qff_qml_benchmark.py",
    "data/q2/hd_data_train.csv",
    "data/q2/hd_data_test_features.csv",
]
for name in required:
    print(f"{name}: {(root / name).exists()}")


train_q2.py: True
predictions_q2.csv: True
processor_d_config.json: True
official_kit/qff_qml_benchmark.py: True
data/q2/hd_data_train.csv: True
data/q2/hd_data_test_features.csv: True


## 2. Review Q2 validation results

In [2]:
import json
from pathlib import Path

metrics_path = Path("results/q2_validation_results.json")
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")),
                     indent=2))
else:
    print("Validation JSON not found. Run train_q2.py and check its output.")


{
  "seed": 42,
  "train_rows": 800,
  "test_rows": 200,
  "validation_rows": 160,
  "validation_results": [
    {
      "model": "rbf_svm_50_features",
      "balanced_accuracy": 0.7106869034579878,
      "f1_macro": 0.7108736643620364
    },
    {
      "model": "rbf_svm_6_features",
      "balanced_accuracy": 0.5385698638710688,
      "f1_macro": 0.4968553459119497
    },
    {
      "model": "quantum_kernel_6_qubits",
      "balanced_accuracy": 0.566499765294946,
      "f1_macro": 0.5658841571310606
    }
  ],
  "selected_model": "rbf_svm_50_features",
  "quantum_kernel_details": null,
  "test_labels_used_for_training_or_selection": false
}


## 3. Inspect prediction file

This cell checks the prediction file's row count and class distribution. It
does not measure test accuracy because the hidden test labels are unavailable.


In [3]:
import pandas as pd

predictions = pd.read_csv("predictions_q2.csv")
display(predictions.head())
print("Rows:", len(predictions))
print("Columns:", predictions.columns.tolist())
print("Prediction counts:")
display(predictions.iloc[:, -1].value_counts(dropna=False))


,sample_id,label
0,s0,A
1,s1,A
2,s2,B
3,s3,A
4,s4,B


Rows: 200
Columns: ['sample_id', 'label']
Prediction counts:


label
B    105
A     95
Name: count, dtype: int64

## 4. Task A — official Bell-state benchmark

The official kit computes ideal and noisy Bell-state fidelity estimates using
8192 shots per correlation. Execute this cell and preserve the printed values
for `results.md`. The official noise model is used without changing its settings.


In [4]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("official_kit").resolve()))
from qff_qml_benchmark import task_a_bell

task_a_results = task_a_bell(shots=8192)
print(json.dumps(task_a_results, indent=2))


{
  "shots": 8192,
  "noise_model_version": "QFF2026-v2",
  "ideal": {
    "XX": 1.0,
    "XX_se": 0.0,
    "YY": -1.0,
    "YY_se": 0.0,
    "ZZ": 1.0,
    "ZZ_se": 0.0,
    "fidelity": 1.0,
    "fidelity_se": 0.0
  },
  "noisy": {
    "XX": 0.819091796875,
    "XX_se": 0.006338134717392098,
    "YY": -0.827880859375,
    "YY_se": 0.0061971720755083825,
    "ZZ": 0.87353515625,
    "ZZ_se": 0.005378000738451436,
    "fidelity": 0.880126953125,
    "fidelity_se": 0.0025920525815950387
  }
}


## 5. Task B — run the official Processor D benchmark

In [5]:
from qff_qml_benchmark import (
    load_processor_d, validate_processor_d, task_b_qft3
)

config = load_processor_d("processor_d_config.json")
design_errors = validate_processor_d(config, require_results=False)
print("Design validation errors:", design_errors)
if design_errors:
    raise ValueError("Processor D design validation failed.")
task_b_results = task_b_qft3(config)
print(json.dumps(task_b_results, indent=2))


Design validation errors: []
{
  "qft3_circuit_depth_before_transpile": 6,
  "qft3_circuit_depth_after_transpile": 22,
  "qft3_two_qubit_gate_count": 12,
  "qft3_swap_overhead": 1,
  "qft3_fidelity_ideal": 1.0,
  "qft3_fidelity_noisy": 0.603585,
  "benchmark_noise_model_version": "QFF2026-v2"
}


## 6. Final Processor D results validation

In [6]:
config = load_processor_d("processor_d_config.json")
validation_errors = validate_processor_d(config, require_results=True)
print("Validation errors:", validation_errors)
print("FINAL VALIDATION:", "PASS" if not validation_errors else "FAIL")


Validation errors: []
FINAL VALIDATION: PASS


## 7. Interpretation and limitations

The recorded validation results favor the 50-feature RBF SVM over the
six-feature baseline and the six-qubit simulated quantum fidelity-kernel
experiment. Validation metrics are not hidden-test performance. The quantum
kernel is simulated; these results do not establish a quantum advantage.
Processor D metrics must be reported from the official benchmark output.
